# The cache cliff

The sieve of Eratosthenes is not compute bound. It is a loop that writes one byte, jumps
forward, and writes another byte. What limits it is how fast memory can feed the CPU, and
that depends almost entirely on whether the array you are striding through fits in cache.

This notebook measures that on **your** machine. It sieves at a range of sizes using two
representations of the same algorithm:

* **one byte per candidate** - a bool array, which is what the starting point uses
* **one bit per odd candidate** - a bitset, sixteen times smaller

then plots the cost of a single "mark this number composite" operation against the sieve
size. If memory did not matter, both lines would be flat.

Run all cells. The measurement takes a minute or two.


In [ ]:
#r "nuget: Plotly.NET.Interactive, 5.0.0"


In [ ]:
# Cache sizes, straight from the hardware.
# Level 3 = L1, Level 4 = L2, Level 5 = L3 in the CIM schema.

$l1Bytes = 0
$l2Bytes = 0
$l3Bytes = 0

try {
    $caches = Get-CimInstance Win32_CacheMemory -ErrorAction Stop
    $caches | Select-Object Purpose, Level, InstalledSize | Format-Table -AutoSize | Out-String | Write-Output

    $l1Bytes = 1024 * (($caches | Where-Object { $_.Level -eq 3 } | Measure-Object InstalledSize -Minimum).Minimum)
    $l2Bytes = 1024 * (($caches | Where-Object { $_.Level -eq 4 } | Measure-Object InstalledSize -Minimum).Minimum)
    $l3Bytes = 1024 * (($caches | Where-Object { $_.Level -eq 5 } | Measure-Object InstalledSize -Maximum).Maximum)

    Write-Output "L1 (smallest reported): $l1Bytes bytes"
    Write-Output "L2 (smallest reported): $l2Bytes bytes"
    Write-Output "L3 (largest reported):  $l3Bytes bytes"
    Write-Output ""
    Write-Output "Note: on a hybrid CPU these are per-cluster totals, not per-core."
    Write-Output "Trust the shape of the curve over these numbers."
}
catch {
    Write-Output "Could not read cache sizes (this query is Windows only)."
    Write-Output "The charts still work, just without the cache reference lines."
}


In [ ]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;

// One byte per candidate. Square root bound, start marking at f*f.
static void SieveBytes(bool[] sieve, int n)
{
    int limit = (int)Math.Sqrt(n);
    for (int f = 2; f <= limit; f++)
        if (sieve[f])
            for (int q = f * f; q <= n; q += f)
                sieve[q] = false;
}

// The identical loop, counting the writes instead of only doing them.
static long CountBytes(bool[] sieve, int n)
{
    long ops = 0;
    int limit = (int)Math.Sqrt(n);
    for (int f = 2; f <= limit; f++)
        if (sieve[f])
            for (int q = f * f; q <= n; q += f) { sieve[q] = false; ops++; }
    return ops;
}

// One bit per odd candidate. Bit i represents the number 2i+1.
static void SieveBits(ulong[] words, int n)
{
    int bits = (n + 1) / 2;
    int limit = (int)Math.Sqrt(n);
    for (int f = 3; f <= limit; f += 2)
    {
        int fi = f >> 1;
        if ((words[fi >> 6] & (1UL << fi)) != 0) continue;
        for (int q = (f * f) >> 1; q < bits; q += f)
            words[q >> 6] |= 1UL << q;
    }
}

static long CountBits(ulong[] words, int n)
{
    long ops = 0;
    int bits = (n + 1) / 2;
    int limit = (int)Math.Sqrt(n);
    for (int f = 3; f <= limit; f += 2)
    {
        int fi = f >> 1;
        if ((words[fi >> 6] & (1UL << fi)) != 0) continue;
        for (int q = (f * f) >> 1; q < bits; q += f) { words[q >> 6] |= 1UL << q; ops++; }
    }
    return ops;
}

// Half octave steps from 8 Ki up to 64 Mi, plus the race size itself.
var sizes = new List<int>();
for (int k = 0; k <= 26; k++) sizes.Add((int)Math.Round(Math.Pow(2, 13 + k / 2.0)));
sizes.Add(1_000_000);
sizes = sizes.Distinct().OrderBy(x => x).ToList();

var nList = new List<double>();
var arrayBytesList = new List<double>();
var bitsetBytesList = new List<double>();
var nsArrayList = new List<double>();
var nsBitsetList = new List<double>();
var msArrayList = new List<double>();
var msBitsetList = new List<double>();

foreach (int n in sizes)
{
    var sieve = new bool[n + 1];
    var words = new ulong[(((n + 1) / 2) + 63) / 64];

    Array.Fill(sieve, true);
    long opsBytes = CountBytes(sieve, n);
    Array.Clear(words);
    long opsBits = CountBits(words, n);

    double Measure(Action reset, Action run)
    {
        // Warm up on a time budget. Tiered JIT needs several calls before it
        // promotes a method, and without this the small sizes read far too slow
        // and flatten the very effect we are trying to show.
        var warm = Stopwatch.StartNew();
        int warmReps = 0;
        while (warm.ElapsedMilliseconds < 100 && warmReps < 30) { reset(); run(); warmReps++; }

        var sw = new Stopwatch();
        double totalMs = 0; int reps = 0;
        while (totalMs < 150 && reps < 2000)
        {
            reset();
            sw.Restart();
            run();
            sw.Stop();
            totalMs += sw.Elapsed.TotalMilliseconds;
            reps++;
        }
        return totalMs / reps;
    }

    double msBytes = Measure(() => Array.Fill(sieve, true), () => SieveBytes(sieve, n));
    double msBits = Measure(() => Array.Clear(words), () => SieveBits(words, n));

    nList.Add(n);
    arrayBytesList.Add(n + 1);
    bitsetBytesList.Add(words.Length * 8);
    nsArrayList.Add(msBytes * 1e6 / opsBytes);
    nsBitsetList.Add(msBits * 1e6 / opsBits);
    msArrayList.Add(msBytes);
    msBitsetList.Add(msBits);

    GC.Collect();
    GC.WaitForPendingFinalizers();
}

double[] ns = nList.ToArray();
double[] arrayBytes = arrayBytesList.ToArray();
double[] bitsetBytes = bitsetBytesList.ToArray();
double[] nsArray = nsArrayList.ToArray();
double[] nsBitset = nsBitsetList.ToArray();
double[] msArray = msArrayList.ToArray();
double[] msBitset = msBitsetList.ToArray();

int race = nList.IndexOf(1_000_000);
Console.WriteLine($"Measured {ns.Length} sizes.");
Console.WriteLine();
Console.WriteLine("At the race size, n = 1,000,000:");
Console.WriteLine($"  byte array {arrayBytes[race] / 1024.0,9:F0} KiB   {nsArray[race]:F3} ns/op   {msArray[race]:F3} ms per sieve");
Console.WriteLine($"  bitset     {bitsetBytes[race] / 1024.0,9:F0} KiB   {nsBitset[race]:F3} ns/op   {msBitset[race]:F3} ms per sieve");


In [ ]:
#!set --name sharedNs --value @csharp:ns
#!set --name sharedNsArray --value @csharp:nsArray
#!set --name sharedNsBitset --value @csharp:nsBitset

open Plotly.NET
open System.Text.Json

// Arrays crossing a kernel boundary arrive as JSON, so unpack them first.
let toFloats (doc: JsonDocument) =
    doc.RootElement.EnumerateArray()
    |> Seq.map (fun e -> e.GetDouble())
    |> Array.ofSeq

let ns = toFloats sharedNs
let nsArray = toFloats sharedNsArray
let nsBitset = toFloats sharedNsBitset

let yMax = (Array.max nsArray) * 1.05

[ Chart.Line(x = ns, y = nsArray, Name = "one byte per candidate", ShowMarkers = true)
  Chart.Line(x = ns, y = nsBitset, Name = "one bit per odd candidate", ShowMarkers = true)
  Chart.Line(x = [| 1e6; 1e6 |], y = [| 0.0; yMax |], Name = "the race (n = 1,000,000)") ]
|> Chart.combine
|> Chart.withXAxisStyle ("sieve size n", AxisType = StyleParam.AxisType.Log)
|> Chart.withYAxisStyle "nanoseconds per marking operation"
|> Chart.withTitle "Cost of one sieve operation as the working set grows"
|> Chart.withSize (900, 500)


In [ ]:
#!set --name sharedNs --value @csharp:ns
#!set --name sharedArrayBytes --value @csharp:arrayBytes
#!set --name sharedBitsetBytes --value @csharp:bitsetBytes
#!set --name l1Bytes --value @pwsh:l1Bytes
#!set --name l2Bytes --value @pwsh:l2Bytes
#!set --name l3Bytes --value @pwsh:l3Bytes

open Plotly.NET
open System.Text.Json

// Same unpacking as the previous cell. Scalars cross natively, arrays come as JSON.
let toFloats (doc: JsonDocument) =
    doc.RootElement.EnumerateArray()
    |> Seq.map (fun e -> e.GetDouble())
    |> Array.ofSeq

let ns = toFloats sharedNs
let arrayBytes = toFloats sharedArrayBytes
let bitsetBytes = toFloats sharedBitsetBytes

// Windows reports these as per-cluster totals, so the L1 and L2 figures are far
// larger than what one core actually gets. Label them honestly rather than
// pretending they are per-core limits. L3 is genuinely shared, so that one is real.
let level (name: string) (bytes: float) =
    let label = sprintf "%s as reported (%.0f KiB)" name (bytes / 1024.0)
    Chart.Line(x = [| Array.min ns; Array.max ns |], y = [| bytes; bytes |], Name = label)

let cacheLines =
    [ "L1", float l1Bytes; "L2", float l2Bytes; "L3", float l3Bytes ]
    |> List.filter (fun (_, b) -> b > 0.0)
    |> List.map (fun (name, b) -> level name b)

[ Chart.Line(x = ns, y = arrayBytes, Name = "one byte per candidate", ShowMarkers = true)
  Chart.Line(x = ns, y = bitsetBytes, Name = "one bit per odd candidate", ShowMarkers = true) ]
@ cacheLines
|> Chart.combine
|> Chart.withXAxisStyle ("sieve size n", AxisType = StyleParam.AxisType.Log)
|> Chart.withYAxisStyle ("working set in bytes", AxisType = StyleParam.AxisType.Log)
|> Chart.withTitle "Why: where each representation falls out of each cache level"
|> Chart.withSize (900, 500)


## What to look for

**The first chart is the point.** The byte array line is not flat. It sits low while the
array fits in cache and climbs as it stops fitting, and by the right hand side each
operation costs several times what it did on the left. Same instructions, same algorithm,
several times the cost. The only thing that changed is how far apart in memory the writes
landed.

**The bit line is much flatter**, and it is flatter for one reason: it is sixteen times
smaller, so it stays inside a given cache level sixteen times longer. Notice that it
starts *higher* than the byte array. A bitset does more work per operation - a read, a
shift, an or, and a write - where the byte array does a single store. It is a straight
trade of instructions for footprint, and it only pays off once you are memory bound.

**Find where the byte array line bends.** Compare that against the second chart and the
cache sizes printed above. Those bends are not noise.

### Questions worth arguing about in your pair

1. At n = 1,000,000, which side of the bend is the byte array on? Is the race even big
   enough for any of this to matter yet?
2. The bitset starts off slower per operation. At what size does it overtake, and why
   would you pick it for the race anyway?
3. The starting point allocates a fresh sieve every lap. Where does that cost show up,
   and is it on either of these charts?
4. If shrinking the data by sixteen times buys this much, what would shrinking it further
   buy? Is there a floor?

### Careful

This measures the marking loop only. Allocating and refilling the array each lap is
excluded, which is deliberate, but it is real cost in the actual race. These charts tell
you about memory behaviour, not about who wins.

Treat the **shape** as the result, not the absolute nanoseconds. Notebook cells run under
a scripting host rather than an optimized Release build, so every number here is somewhat
higher than the same code would be in the race, and the cheapest sizes suffer most. The
ratio between the left of the chart and the right is the honest part.

The cache lines in the second chart come from Windows and, on a hybrid CPU, are totals for
a whole cluster of cores rather than what one core gets to itself. A single P core on a
recent Intel part has more like 48 KiB of L1 data cache and one to two MiB of L2. The L3
figure is genuinely shared and genuinely that big. Where the first chart bends is better
evidence than any of these numbers.
